# <center> Machine Learning in Computational Biology - Term Project </center>

## <center> Task B.3 </center>

### <center> Mathematical Approach: Cell-Conditioned Ligand-Receptor Communication Proxies </center>

---

# Libraries & Paths

In [1]:
############################################################################
import os, warnings, urllib.request
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import anndata as ad
import seaborn as sns
from pathlib import Path
import datetime
############################################################################

SEED = 42
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
############################################################################

data_dir     = Path('../data')
output_dir   = Path('../outputs')
figures_dir  = Path('../figures')
external_dir = data_dir / 'external'
external_dir.mkdir(parents=True, exist_ok=True)

pathway_anndata = output_dir / 'filtered_anndata_pathways.h5ad'
############################################################################

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
print("All set.")

All set.


# Reload the AnnData object produced by the Pathway Scoring Analysis

In [2]:
# Must have the output anndata from the pathway scoring analysis (filtered_anndata_pathways.h5ad)
adata = sc.read_h5ad(pathway_anndata)

# required_cols = {"patient_id","patient_timepoint","response_label"}
# missing_cols = required_cols - set(adata.obs.columns)
# if missing_cols:
#   raise KeyError(f"Missing required columns in adata.obs: {sorted(missing_cols)}")

print(f"{adata.shape}")
print(f"Patients: {adata.obs['patient_id'].nunique()}")
print(f"Samples: {adata.obs['patient_timepoint'].nunique()}")


(16291, 10019)
Patients: 32
Samples: 48


# 3. LR Panel Construction

In [ ]:
# In this step we are loading the CellPhoneDB ligand-receptor interaction data
# The CellPhoneDB data are retrieved from the current GitHub 'master' branch; 
# therefore, this URL does corresponds to the fixed CellPhoneDB release/version, which has now updated to release v5 db.
# Once downloaded, the files are cached locally and reused in subsequent runs.
# All results in this analysis use the same locally cached CellPhoneDB
# snapshot from the time of download, unless these files are deleted or replaced.

CPDB_BASE_URL = ( "https://raw.githubusercontent.com/ventolab/cellphonedb-data/master/data")

for fname in ["interaction_input.csv", "gene_input.csv"]:
    local_path = external_dir / f"cpdb_{fname}"

    if not local_path.exists():
        urllib.request.urlretrieve(f"{CPDB_BASE_URL}/{fname}", local_path)
        print(f"Downloaded: {fname}")

    else:
        # Local file modification time is reported as additional provenance information for the cached snapshot.
        mtime = datetime.datetime.fromtimestamp(local_path.stat().st_mtime)
        print(f"Using cached: {local_path} (local modification time: {mtime})")



# Reading CellPhoneDB interaction and gene annotation tables
interactions_db = pd.read_csv(external_dir / "cpdb_interaction_input.csv")
genes_db = pd.read_csv(external_dir / "cpdb_gene_input.csv")


# CellPhoneDB interaction partners are represented by UniProt
# identifiers, so therefore they are mapped to HGNC gene symbols so that
# they can later be matched to genes present in the AnnData.
uniprot_to_gene = (genes_db.dropna(subset=["uniprot", "hgnc_symbol"])
                   .drop_duplicates(subset="uniprot")
                   .set_index("uniprot")["hgnc_symbol"].to_dict())



# Selecting directed Ligand-Receptor interactions from CellPhoneDB
# Interaction direction obtain from Cellphone DB: partner_a -> ligand, partner_b -> receptor
lr_all = interactions_db[interactions_db["directionality"] == "Ligand-Receptor"].copy()
lr_all["ligand"] = (lr_all["partner_a"].map(uniprot_to_gene))
lr_all["receptor"] = (lr_all["partner_b"].map(uniprot_to_gene))

# Retaining simple gene-to-gene Ligand-Receptor interactions (no molecular complexes) 
lr_simple = (lr_all.dropna(subset=["ligand", "receptor"]).drop_duplicates(subset=["ligand", "receptor"]).copy())
print(f"Simple (non-complex) Ligand-Receptor pairs in CellPhoneDB: {len(lr_simple)}")

Using cached: ..\data\external\cpdb_interaction_input.csv (local modification time: 2026-09-10 01:30:52.132935)
Using cached: ..\data\external\cpdb_gene_input.csv (local modification time: 2026-09-10 01:30:52.504303)
Simple (non-complex) Ligand-Receptor pairs in CellPhoneDB: 770


### Part A - Selection of immune/TME-related Ligand-Receptor pairs

In [4]:
# Aim --> include more immune-related interactions as a prior step of the analysis.
# We retrieve CellPhoneDB interactions that belong to immune- and tumor microenvironment (TME)-related categories.

# Must check the available CellPhoneDB interaction classifications in order to select the biologically relevant immune/TME categories.
print(lr_simple['classification'].value_counts().sort_index().to_string())

classification
Adhesion by Cadherin                                                2
Adhesion by ICAM                                                    4
Adhesion by Prothrombin                                             4
Signaling by Adrenomedullin                                         1
Signaling by Agouti-related protein                                 5
Signaling by Agrin                                                  1
Signaling by Amylin                                                 1
Signaling by Amyloid-beta precursor protein                         5
Signaling by Amyloid-like protein                                   2
Signaling by Angiopoietin                                           3
Signaling by Angiotensinogen                                        2
Signaling by Annexin                                                3
Signaling by Apelin                                                 2
Signaling by Apolipoprotein                                         2
Signa

In [5]:
# Categories are defined based on their biological relevance to the
# immune/TME context and independently of the response labels.

IMMUNE_TME_CATEGORIES = ["Signaling by Chemokines","Signaling by Interleukin",
                         "Signaling by Tumor necrosis factor","Signaling by HLA",
                         "Signaling by Transforming growth factor","Signaling by Inhibin/Activin",
                         "Signaling by Lipoxin/Leukotriene","Signaling by Complement",
                         "Signaling by Notch","Signaling by Colony-Stimulating factor",
                         "Signaling by Poliovirus receptor","Signaling by Nectin",
                         "Signaling by Galectin","Signaling by Vascular endothelial growth factor",
                         "Signaling by Lymphotactin","Signaling by Pro-MHC",
                         "Adhesion by ICAM","Signaling by Selectin"]

# Keeping only LR pairs that belong to the selected immune/TME categories & 
# the source of each selected pair for later documentation.
category_pairs = lr_simple[lr_simple["classification"].isin(IMMUNE_TME_CATEGORIES)].copy()
category_pairs["panel_source"] = "immune_category"

print( f"Part A -- immune/TME category-tagged pairs: {len(category_pairs)}")

Part A -- immune/TME category-tagged pairs: 217


### Part B - Additional ICI-relevant Ligand-Receptor pairs

In [ ]:
# Five well-established immune checkpoint and co-stimulatory interactions are added a-priori,
# since their CellPhoneDB classification is NA and therefore they are not captured by the
# immune/TME category-based selection above, although they are biologically relevant to the immune context.
# These interactions, as well as the prior category choices, are defined independently
# of the response labels in order to avoid information leakage in the later modeling steps.

NAMED_LOOKUP_PAIRS = [("CD274", "PDCD1"), ("CD86", "CTLA4"), ("CD86", "CD28"), ("ICOSLG", "ICOS"),("CD40LG", "CD40")] # [16]
 
named_pairs_rows = []
for ligand, receptor in NAMED_LOOKUP_PAIRS:

    match = lr_simple[(lr_simple["ligand"] == ligand) & (lr_simple["receptor"] == receptor)]

    # Keeping the pair only if the same ligand -> receptor direction is present in CellPhoneDB.
    if match.empty:
        print(f"WARNING - Skip: {ligand} -> {receptor} not found as a directed Ligand-Receptor interaction in CellPhoneDB.")
        continue

    named_pairs_rows.append(match.iloc[0])


named_pairs = pd.DataFrame(named_pairs_rows)

# Recording that these interactions were added through the
# predefined ICI-related lookup rather than the category selection, for possible future change, by the CellPhoneDB 
if not named_pairs.empty:
    named_pairs["panel_source"] = "named_lookup"

print(f"Part B -- named-lookup pairs found: {len(named_pairs)} / {len(NAMED_LOOKUP_PAIRS)}")

if not named_pairs.empty:
    display(named_pairs[["ligand", "receptor", "classification", "panel_source"]])


Part B -- named-lookup pairs found: 5 / 5


,ligand,receptor,classification,panel_source
2791,CD274,PDCD1,NaN,named_lookup
2810,CD86,CTLA4,NaN,named_lookup
2809,CD86,CD28,NaN,named_lookup
2839,ICOSLG,ICOS,NaN,named_lookup
2796,CD40LG,CD40,NaN,named_lookup


### **Union of Parts A and B, deduplicated:**

In [ ]:
# Combining the immune/TME category-based pairs (Part A)
# with the additional ICI-related pairs (Part B) and deduplicating.

combined_panel = (pd.concat([category_pairs, named_pairs],ignore_index=True)
                  .drop_duplicates(subset=["ligand", "receptor"], keep="first")
                  .reset_index(drop=True))

print(f"Combined candidate panel (before AnnData gene-overlap filter): {len(combined_panel)}")
print(combined_panel["panel_source"].value_counts().to_string())


Combined candidate panel (before AnnData gene-overlap filter): 222
panel_source
immune_category    217
named_lookup         5


# Gene-Overlap Filter

In [8]:
# Keeping only Ligand-Receptor pairs for which both genes are present in the filtered AnnData gene set.

adata_genes = set(adata.var_names.astype(str))

valid_rows = []
dropped_pairs = []


for _, row in combined_panel.iterrows():

    ligand = row["ligand"]
    receptor = row["receptor"]
    ligand_present = ligand in adata_genes
    receptor_present = receptor in adata_genes

    if ligand_present and receptor_present:
        valid_rows.append(row.copy())

    else:
        missing_side = []

        if not ligand_present:
            missing_side.append(f"ligand {ligand} missing")
        if not receptor_present:
            missing_side.append(f"receptor {receptor} missing")


        dropped_pairs.append(
            {"ligand": ligand, "receptor": receptor,
             "panel_source": row.get("panel_source",np.nan),
             "classification": row.get("classification",np.nan),
             "reason": "; ".join(missing_side)})



# Final valid and dropped LR panels 
valid_pairs_df = (pd.DataFrame(valid_rows).reset_index(drop=True))
valid_pairs = list(valid_pairs_df[["ligand", "receptor"]].itertuples(index=False,name=None))
dropped_pairs_df = pd.DataFrame(dropped_pairs)

print(f"Candidate pairs before gene-overlap filter: {len(combined_panel)}")
print(f"Final valid pairs (both genes present): {len(valid_pairs)}")
print(f"Dropped (gene missing from AnnData): {len(dropped_pairs_df)}")


display(dropped_pairs_df)

Candidate pairs before gene-overlap filter: 222
Final valid pairs (both genes present): 55
Dropped (gene missing from AnnData): 167


,ligand,receptor,panel_source,classification,reason
0,ICAM2,CD209,immune_category,Adhesion by ICAM,receptor CD209 missing
1,ICAM3,CD209,immune_category,Adhesion by ICAM,receptor CD209 missing
2,ICAM3,CLEC4M,immune_category,Adhesion by ICAM,receptor CLEC4M missing
3,CCL1,CCR8,immune_category,Signaling by Chemokines,ligand CCL1 missing
4,CCL11,ACKR2,immune_category,Signaling by Chemokines,ligand CCL11 missing; receptor ACKR2 missing
...,...,...,...,...,...
162,VEGFD,KDR,immune_category,Signaling by Vascular endothelial growth factor,ligand VEGFD missing; receptor KDR missing
163,INHBC,ACVR2A,immune_category,Signaling by Inhibin/Activin,ligand INHBC missing
164,INHBC,ACVR2B,immune_category,Signaling by Inhibin/Activin,ligand INHBC missing; receptor ACVR2B missing
165,INHBE,ACVR2A,immune_category,Signaling by Inhibin/Activin,ligand INHBE missing


## Sample-level mean expression cell

In [ ]:
# Extracting the mean expression of all genes involved in the final LR panel for observation purposes
# and for later calculation and inspection of the ligand-receptor scoring results.

needed_genes = sorted({gene for ligand, receptor in valid_pairs for gene in (ligand, receptor)})

X_needed = adata[:, needed_genes].X
if sp.issparse(X_needed):
    X_needed = X_needed.toarray()

# Converting to DataFrame so that expression can be grouped
# by biological sample (patient_timepoint).
X_needed = pd.DataFrame(X_needed,index=adata.obs_names,columns=needed_genes)
X_needed["patient_timepoint"] = (adata.obs["patient_timepoint"].values)

# Mean expression of each LR gene within each biological sample
sample_mean_expr = (X_needed.groupby("patient_timepoint", observed=True)[needed_genes].mean())

print(f"Per-gene sample-level mean expression matrix shape: {sample_mean_expr.shape}")
print(f"Unique genes involved (ligands + receptors): {len(needed_genes)}")

display(sample_mean_expr.head(10))

Per-gene sample-level mean expression matrix shape: (48, 77)
Unique genes involved (ligands + receptors): 77


,BTLA,CCL2,CCL3,CCL3L1,CCL4,CCL5,CCR1,CCR2,CCR4,CCR5,...,TNFRSF25,TNFRSF4,TNFRSF9,TNFSF10,TNFSF12,TNFSF14,TNFSF4,TNFSF9,VEGFA,VEGFB
patient_timepoint,,,,,,,,,,,,,,,,,,,,,
Post_P1,1.037526,0.000000,1.479553,0.192852,2.741340,7.586632,0.190481,0.242818,0.271478,1.068935,...,1.604433,1.511546,2.891375,2.032749,0.345395,1.776083,0.178076,2.391615,0.062268,0.251856
Post_P1_2,1.768187,0.000000,2.164334,0.439320,5.157479,7.757762,0.337819,0.422918,0.570085,1.636006,...,1.661586,1.659433,2.887762,1.247989,0.508867,0.335694,0.793484,0.728130,0.047819,0.219547
Post_P2,1.064605,0.000000,3.804947,1.096421,5.445711,7.956421,0.730000,0.623658,0.712816,2.673790,...,2.065184,1.336763,2.578079,1.577842,0.356605,0.456289,0.764737,2.033500,0.069105,0.344079
Post_P3,0.445515,0.203733,2.056602,0.793816,5.582033,8.067298,0.468914,0.672618,0.716295,2.615961,...,2.106546,1.363203,1.777771,1.739638,0.341811,0.657883,0.455376,1.919387,0.242145,0.355460
Post_P3_2,0.282346,0.024274,0.959972,0.472263,3.957905,8.884107,0.101564,0.142039,0.450475,1.133520,...,1.708575,0.677849,1.249972,0.768408,0.084469,0.596955,0.224581,2.706592,0.123128,0.146844
Post_P4,2.012726,0.012163,0.435244,0.271852,1.232711,2.484548,0.068015,0.196948,0.555230,0.708726,...,1.766341,0.701689,0.137037,0.798059,0.424993,0.218311,0.122119,0.619407,0.005911,0.477630
Post_P5,0.437295,0.514897,2.561644,0.942740,5.588699,8.048904,0.464075,0.410959,0.314110,2.055308,...,1.382979,0.588733,2.782021,1.067911,0.581541,0.300308,1.473253,1.445000,0.327911,0.283425
Post_P5_2,0.360833,0.000000,1.502500,0.379315,6.683125,9.585863,0.196310,0.331518,0.439702,1.432708,...,1.243184,0.584762,3.036428,0.299881,0.127649,0.095208,0.843809,1.986458,0.029583,0.166250
Post_P6,1.164350,0.252550,4.555000,0.925250,5.681625,6.418575,0.922475,0.542150,0.631350,2.807150,...,1.300775,1.570850,2.321875,3.006200,0.392450,0.138125,0.391900,1.506100,0.233850,0.877050


In [ ]:
# Sample-level Ligand-Receptor co-expression scores

# For each biological sample and each valid LR pair, the score is calculated as:
# mean ligand expression in the sample × mean receptor expression in the same sample.

ccc_scores = pd.DataFrame(index=sample_mean_expr.index)

for ligand, receptor in valid_pairs:
    ccc_scores[f"CCC_{ligand}_to_{receptor}"] = (sample_mean_expr[ligand] * sample_mean_expr[receptor])

print(f"Sample-level LR score matrix shape: {ccc_scores.shape}")

display(ccc_scores.head(10))

Sample-level LR score matrix shape:  (48, 55)


,CCC_ICAM1_to_ITGAL,CCC_CCL2_to_CCR2,CCC_CCL3_to_CCR1,CCC_CCL3_to_CCR5,CCC_CCL3L1_to_CCR1,CCC_CCL4_to_CCR5,CCC_CCL5_to_CCR1,CCC_CCL5_to_CCR4,CCC_CCL5_to_CCR5,CCC_CXCL13_to_CXCR5,...,CCC_TNFSF4_to_TNFRSF4,CCC_TNFSF9_to_TNFRSF9,CCC_VEGFA_to_NRP1,CCC_VEGFA_to_NRP2,CCC_VEGFB_to_NRP1,CCC_CD274_to_PDCD1,CCC_CD86_to_CTLA4,CCC_CD86_to_CD28,CCC_ICOSLG_to_ICOS,CCC_CD40LG_to_CD40
patient_timepoint,,,,,,,,,,,,,,,,,,,,,
Post_P1,10.634980,0.000000,0.281827,1.581546,0.036735,2.930314,1.445110,2.059601,8.109614,2.582389,...,0.269170,6.915055,0.003223,0.007774,0.013034,2.505798,0.738486,0.711264,0.553488,0.139406
Post_P1_2,4.412673,0.000000,0.731153,3.540863,0.148411,8.437664,2.620717,4.422584,12.691742,2.443894,...,1.316735,2.102667,0.025156,0.000340,0.115495,1.732334,0.631688,0.791856,0.134790,0.194594
Post_P2,6.437128,0.000000,2.777611,10.173629,0.800387,14.560684,5.808187,5.671462,21.273794,2.467329,...,1.022272,5.242524,0.014354,0.001108,0.071469,3.092469,2.216768,1.275027,0.322274,0.213249
Post_P3,6.313454,0.137034,0.964369,5.379990,0.372231,14.602382,3.782866,5.778567,21.103737,0.856846,...,0.620770,3.412232,0.068212,0.086902,0.100133,1.150532,1.662281,1.102467,0.346713,0.141600
Post_P3_2,5.029674,0.003448,0.097499,1.088147,0.047965,4.486363,0.902308,4.002067,10.070309,0.457247,...,0.152232,3.383165,0.034276,0.011518,0.040878,0.533651,0.784544,0.978532,0.647359,0.071091
Post_P4,4.350281,0.002395,0.029603,0.308469,0.018490,0.873654,0.168986,1.379495,1.760864,1.185825,...,0.085689,0.084882,0.002049,0.000197,0.165599,0.180138,0.716872,1.135135,1.216204,1.736961
Post_P5,5.279485,0.211602,1.188796,5.264968,0.437502,11.486500,3.735298,2.528238,16.542980,0.795691,...,0.867353,4.020020,0.127301,0.090838,0.110031,1.535911,6.071759,1.674267,0.716364,0.128574
Post_P5_2,1.718907,0.000000,0.294955,2.152644,0.074463,9.574969,1.881796,4.214927,13.733747,3.431467,...,0.493428,6.031739,0.000000,0.000711,0.000000,0.851967,1.535289,0.263597,0.087764,0.023405
Post_P6,14.787270,0.136920,4.201873,12.786568,0.853520,15.949173,5.920975,4.052367,18.017902,0.608557,...,0.615616,3.496976,0.164326,0.012096,0.616303,3.079490,2.330059,3.424718,0.506819,0.428802


**Comment:** The above calculation provides a sample-level LR co-expression proxy. However, it is not used as the final cell-level CCC representation, because each sample has only one score for each LR pair. Adding these scores to AnnData at the cell level would therefore repeat the same sample-level values across all cells belonging to the same sample.

For this reason, the sample-level scores are kept as an intermediate representation, while the next step introduces cell-conditioned sender-like and receiver-like scores that preserve variation between individual cells.

- Worth mentioning: None of the representations consist of a direct cell-cell communication.
---

## Cell-specific CCC scoring

### Cell-conditioned Sender-like and Receiver-like Ligand-Receptor scores

These cell-specific scores are cell-conditioned ligand–receptor co-expression proxies, meaning that they do not identify a physical sender–receiver cell pair and do not demonstrate direct cell–cell communication.

The sender-like score combines ligand expression in an individual cell with the mean receptor expression of the biological sample to which that cell belongs:

$$S^{\mathrm{sender}}_{c,L\rightarrow R} = x_{c,L}\cdot \overline{x}_{s(c),R}$$

The receiver-like score combines the mean ligand expression of the biological sample with receptor expression in the individual cell:

$$S^{\mathrm{receiver}}_{c,L\rightarrow R} = \overline{x}_{s(c),L}\cdot x_{c,R}$$

Therefore, these scores describe how strongly each individual cell contributes to a sender-like or receiver-like ligand–receptor pattern within the expression context of its own sample.

In [ ]:
cell_specific_ccc = pd.DataFrame(index=X_needed.index)

for ligand, receptor in valid_pairs:

    # Sample-level mean expression mapped back to each cell
    sample_mean_ligand = (X_needed["patient_timepoint"].map(sample_mean_expr[ligand]).astype(float))
    sample_mean_receptor = (X_needed["patient_timepoint"].map(sample_mean_expr[receptor]).astype(float))

    # Cell-level ligand and receptor expression
    cell_ligand = X_needed[ligand].astype(float)
    cell_receptor = X_needed[receptor].astype(float)

    # Sender-like score:
    cell_specific_ccc[ f"CCC_Sender_{ligand}_to_{receptor}"] = (cell_ligand * sample_mean_receptor)

    # Receiver-like score:
    cell_specific_ccc[f"CCC_Receiver_{ligand}_to_{receptor}"] = (sample_mean_ligand * cell_receptor)

In [12]:
expected_features = 2 * len(valid_pairs)

# Additional prints
print("Cell-conditioned CCC proxy matrix")
print(f"Cells: {cell_specific_ccc.shape[0]}")
print(f"Valid LR pairs: {len(valid_pairs)}")
print(f"Expected CCC features: {expected_features}")
print(f"Actual CCC features: {cell_specific_ccc.shape[1]}")
print("Missing values:",int(cell_specific_ccc.isna().sum().sum()))
print("Infinite values:", int(np.isinf(cell_specific_ccc.to_numpy(dtype=float)).sum()))

print("All columns numeric:", all( pd.api.types.is_numeric_dtype(cell_specific_ccc[col] ) 
                                  for col in cell_specific_ccc.columns))


# Just Checking
if cell_specific_ccc.shape[1] != expected_features:
    raise ValueError( "Unexpected number of cell-conditioned CCC features.")

if cell_specific_ccc.isna().any().any():
    raise ValueError("NaN values found in cell-conditioned CCC scores.")

if not np.isfinite(cell_specific_ccc.to_numpy(dtype=float)).all():
    raise ValueError("Non-finite values found in cell-conditioned CCC scores."
    )

Cell-conditioned CCC proxy matrix
Cells: 16291
Valid LR pairs: 55
Expected CCC features: 110
Actual CCC features: 110
Missing values: 0
Infinite values: 0
All columns numeric: True


In [13]:
# Previews
display(cell_specific_ccc.head())
cell_specific_preview = (adata.obs[["patient_timepoint"]].join(cell_specific_ccc))
display( cell_specific_preview.head(20))

,CCC_Sender_ICAM1_to_ITGAL,CCC_Receiver_ICAM1_to_ITGAL,CCC_Sender_CCL2_to_CCR2,CCC_Receiver_CCL2_to_CCR2,CCC_Sender_CCL3_to_CCR1,CCC_Receiver_CCL3_to_CCR1,CCC_Sender_CCL3_to_CCR5,CCC_Receiver_CCL3_to_CCR5,CCC_Sender_CCL3L1_to_CCR1,CCC_Receiver_CCL3L1_to_CCR1,...,CCC_Sender_CD274_to_PDCD1,CCC_Receiver_CD274_to_PDCD1,CCC_Sender_CD86_to_CTLA4,CCC_Receiver_CD86_to_CTLA4,CCC_Sender_CD86_to_CD28,CCC_Receiver_CD86_to_CD28,CCC_Sender_ICOSLG_to_ICOS,CCC_Receiver_ICOSLG_to_ICOS,CCC_Sender_CD40LG_to_CD40,CCC_Receiver_CD40LG_to_CD40
cell_id,,,,,,,,,,,,,,,,,,,,,
A10_P3_M11,45.070113,12.642813,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
A11_P1_M11,0.000000,12.013817,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,1.819246,0.0,0.0
A11_P3_M11,0.000000,10.692926,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.547028,0.0,0.000000,0.0,2.054992,0.0,0.0
A11_P4_M11,0.000000,10.768406,0.0,0.218555,0.0,0.0,0.0,8.235495,0.0,0.0,...,20.584421,3.395629,0.0,0.501046,0.0,1.626815,0.0,2.255154,0.0,0.0
A12_P3_M11,0.000000,6.478655,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0


,patient_timepoint,CCC_Sender_ICAM1_to_ITGAL,CCC_Receiver_ICAM1_to_ITGAL,CCC_Sender_CCL2_to_CCR2,CCC_Receiver_CCL2_to_CCR2,CCC_Sender_CCL3_to_CCR1,CCC_Receiver_CCL3_to_CCR1,CCC_Sender_CCL3_to_CCR5,CCC_Receiver_CCL3_to_CCR5,CCC_Sender_CCL3L1_to_CCR1,...,CCC_Sender_CD274_to_PDCD1,CCC_Receiver_CD274_to_PDCD1,CCC_Sender_CD86_to_CTLA4,CCC_Receiver_CD86_to_CTLA4,CCC_Sender_CD86_to_CD28,CCC_Receiver_CD86_to_CD28,CCC_Sender_ICOSLG_to_ICOS,CCC_Receiver_ICOSLG_to_ICOS,CCC_Sender_CD40LG_to_CD40,CCC_Receiver_CD40LG_to_CD40
cell_id,,,,,,,,,,,,,,,,,,,,,
A10_P3_M11,Pre_P1,45.070113,12.642813,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0
A11_P1_M11,Pre_P1,0.000000,12.013817,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,1.819246,0.000000,0.0
A11_P3_M11,Pre_P1,0.000000,10.692926,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.0,0.547028,0.0,0.000000,0.000000,2.054992,0.000000,0.0
A11_P4_M11,Pre_P1,0.000000,10.768406,0.0,0.218555,0.000000,0.0,0.000000,8.235495,0.0,...,20.584421,3.395629,0.0,0.501046,0.0,1.626815,0.000000,2.255154,0.000000,0.0
A12_P3_M11,Pre_P1,0.000000,6.478655,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0
A12_P6_M11,Pre_P1,0.000000,11.045163,0.0,0.000000,0.736357,0.0,8.472565,9.033509,0.0,...,0.000000,4.352145,0.0,0.000000,0.0,1.247859,0.000000,0.000000,0.000000,0.0
A2_P1_M11,Pre_P1,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,1.266886,0.000000,2.528708,0.000000,0.0
A2_P4_M11,Pre_P1,25.586677,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.000000,4.862286,0.0,0.000000,0.0,0.000000,25.975438,0.000000,0.000000,0.0
A3_P1_M11,Pre_P1,43.286087,11.661580,0.0,0.000000,0.000000,0.0,0.000000,8.501500,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0


## Saving the new Data in the Anndata

In [ ]:
adata_cellccc = adata.copy()

# Aligning CCC rows exactly to the AnnData cell order
cell_specific_ccc_aligned = (cell_specific_ccc.reindex(adata_cellccc.obs_names))

if cell_specific_ccc_aligned.isna().any().any():
    raise ValueError("NaN values found after aligning CCC scores with AnnData cells.")

if not cell_specific_ccc_aligned.index.equals(adata_cellccc.obs_names):
    raise ValueError("CCC rows are not aligned with AnnData cells.")


# Storing the cell-conditioned CCC matrix (obsm)
adata_cellccc.obsm["cell_specific_ccc"] = (cell_specific_ccc_aligned.to_numpy(dtype=np.float32))

# Storing the corresponding feature names (uns)
adata_cellccc.uns["cell_specific_ccc_feature_names"] = (cell_specific_ccc_aligned.columns.astype(str).tolist())

# Saving the updated AnnData object
# Needed for pandas nullable string columns in obs
ad.settings.allow_write_nullable_strings = True

output_path_cellccc = (output_dir/ "filtered_anndata_pathways_cellSpecificCCC.h5ad")
adata_cellccc.write_h5ad(output_path_cellccc)

In [15]:
# Confirmation step

print("Saved successfully:")
print(output_path_cellccc)
print("\nAnnData dimensions:")
print("Cells:", adata_cellccc.n_obs)
print("Genes:", adata_cellccc.n_vars)
print("PROGENy features:",len([col for col in adata_cellccc.obs.columns if col.startswith("PROGENy_")]))
print("Cell-conditioned CCC matrix:",adata_cellccc.obsm["cell_specific_ccc"].shape)
print("Cell-conditioned CCC feature names:",len(adata_cellccc.uns["cell_specific_ccc_feature_names"]))

Saved successfully:
..\outputs\filtered_anndata_pathways_cellSpecificCCC.h5ad

AnnData dimensions:
Cells: 16291
Genes: 10019
PROGENy features: 14
Cell-conditioned CCC matrix: (16291, 110)
Cell-conditioned CCC feature names: 110


### Verifying CCC Scoring was successfully saved

In [16]:
# Reloading the saved AnnData file from disk
adata_cellccc_check = sc.read_h5ad(output_path_cellccc)

print("\nOBSM keys:")
print(list(adata_cellccc_check.obsm.keys()))

print("\nUNS keys:")
print(list(adata_cellccc_check.uns.keys()))

# Checking that the CCC matrix and feature names were saved
if "cell_specific_ccc" not in adata_cellccc_check.obsm:
    raise ValueError("cell_specific_ccc was NOT found in saved AnnData.obsm")

if "cell_specific_ccc_feature_names" not in adata_cellccc_check.uns:
    raise ValueError("cell_specific_ccc_feature_names was NOT found in saved AnnData.uns")


ccc_saved = (adata_cellccc_check.obsm["cell_specific_ccc"])
ccc_feature_names_saved = (adata_cellccc_check.uns["cell_specific_ccc_feature_names"])

print("\nSaved cell-specific CCC shape:",ccc_saved.shape)
print("Saved CCC feature names:",len(ccc_feature_names_saved))
print("\nFirst 5 CCC feature names:")
print(ccc_feature_names_saved[:5])



OBSM keys:
['X_pca', 'X_umap', 'cell_specific_ccc', 'padj_mlm', 'score_mlm']

UNS keys:
['cell_specific_ccc_feature_names', 'neighbors', 'pca', 'response_label_colors', 'umap']

Saved cell-specific CCC shape: (16291, 110)
Saved CCC feature names: 110

First 5 CCC feature names:
['CCC_Sender_ICAM1_to_ITGAL' 'CCC_Receiver_ICAM1_to_ITGAL'
 'CCC_Sender_CCL2_to_CCR2' 'CCC_Receiver_CCL2_to_CCR2'
 'CCC_Sender_CCL3_to_CCR1']


---
# References
> [16]. Selection of the additional immune/TME-related Ligand-Receptor pairs: https://www.cellphonedb.org